# Credit Card Approval Prediction System - Exploratory Notebook

This notebook guides you through the process of developing a machine learning pipeline to predict credit card application approvals. We will cover:
1. Data Loading and Inspection
2. Label Engineering and Dataset Merging
3. Exploratory Data Analysis (EDA)
4. Data Cleaning
5. Feature Engineering
6. Data Preprocessing (Scaling & Encoding)
7. Model Training & Comparison
8. Saving the Best Model

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Set styling for graphs
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
print("Libraries imported and plotting configurations set.")

In [ ]:
# Step 1: Ingest raw datasets
print("Loading datasets...")
app_df = pd.read_csv('data/application_record.csv')
credit_df = pd.read_csv('data/credit_record.csv')

print(f"Application record dimensions: {app_df.shape}")
print(f"Credit record dimensions: {credit_df.shape}")

In [ ]:
# Display basic information and check missing values/duplicates
print("--- Application Record Info ---")
print(app_df.info())
print("\n--- Missing Values in Application Record ---")
print(app_df.isnull().sum())
print(f"\nDuplicate rows in Application Record: {app_df.duplicated().sum()}")

print("\n--- Credit Record Info ---")
print(credit_df.info())
print("\n--- Missing Values in Credit Record ---")
print(credit_df.isnull().sum())
print(f"\nDuplicate rows in Credit Record: {credit_df.duplicated().sum()}")

In [ ]:
# Step 2: Target Creation and Merging
# We label status '2', '3', '4', or '5' (60+ days overdue) as bad pay history (risky = 1)
credit_df['is_risky'] = credit_df['STATUS'].isin(['2', '3', '4', '5']).astype(int)

# Group by ID to find if a customer defaulted at least once in their history
user_target = credit_df.groupby('ID')['is_risky'].max().reset_index()

# Approved is 1, Rejected is 0
user_target['target'] = 1 - user_target['is_risky']
user_target.drop(columns=['is_risky'], inplace=True)

# Merge application details with their targets using inner join on ID
df = pd.merge(app_df, user_target, on='ID', how='inner')
print(f"Merged Dataset Dimensions: {df.shape}")
print("\nTarget Class Distribution:")
print(df['target'].value_counts(normalize=True) * 100)

In [ ]:
# Step 3: Exploratory Data Analysis (EDA)
# Class distribution chart
plt.figure(figsize=(6, 4))
sns.countplot(x='target', data=df, hue='target', palette='viridis', legend=False)
plt.title('Target Variable Distribution (0 = Rejected, 1 = Approved)')
plt.xlabel('Approval Status')
plt.ylabel('Count')
plt.show()

In [ ]:
# Demographic Plots
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

sns.countplot(ax=axes[0, 0], x='CODE_GENDER', hue='target', data=df, palette='Set2')
axes[0, 0].set_title('Gender Distribution by Approval Status')

sns.countplot(ax=axes[0, 1], x='FLAG_OWN_CAR', hue='target', data=df, palette='Set2')
axes[0, 1].set_title('Car Ownership by Approval Status')

sns.countplot(ax=axes[1, 0], x='FLAG_OWN_REALTY', hue='target', data=df, palette='Set2')
axes[1, 0].set_title('Property Ownership by Approval Status')

sns.countplot(ax=axes[1, 1], x='NAME_FAMILY_STATUS', hue='target', data=df, palette='Set2')
axes[1, 1].set_title('Family Status by Approval Status')
axes[1, 1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
# Education and Housing Analyses
fig, axes = plt.subplots(2, 1, figsize=(12, 12))

sns.countplot(ax=axes[0], x='NAME_EDUCATION_TYPE', hue='target', data=df, palette='Set1')
axes[0].set_title('Education Level by Approval Status')
axes[0].tick_params(axis='x', rotation=15)

sns.countplot(ax=axes[1], x='NAME_HOUSING_TYPE', hue='target', data=df, palette='Set1')
axes[1].set_title('Housing Type by Approval Status')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

In [ ]:
# Age and Employment Years Calculation
df['AGE'] = (df['DAYS_BIRTH'] / -365.25).astype(int)
df['YEARS_EMPLOYED'] = df['DAYS_EMPLOYED'].apply(lambda x: 0 if x > 0 else x / -365.25)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.histplot(ax=axes[0], data=df, x='AGE', hue='target', multiple='stack', bins=30, palette='crest')
axes[0].set_title('Age Distribution in Years')

sns.histplot(ax=axes[1], data=df, x='YEARS_EMPLOYED', hue='target', multiple='stack', bins=30, palette='magma')
axes[1].set_title('Employment Duration in Years')

plt.tight_layout()
plt.show()

In [ ]:
# Occupation Analysis and Income Distribution
fig, axes = plt.subplots(2, 1, figsize=(14, 12))

sns.countplot(ax=axes[0], x='OCCUPATION_TYPE', hue='target', data=df, palette='tab20')
axes[0].set_title('Occupation Distribution by Approval Status')
axes[0].tick_params(axis='x', rotation=45)

sns.kdeplot(ax=axes[1], data=df, x='AMT_INCOME_TOTAL', hue='target', fill=True, common_norm=False, palette='coolwarm')
axes[1].set_title('Income Probability Density')
axes[1].set_xlim(0, 800000) # clip display range for readability

plt.tight_layout()
plt.show()

In [ ]:
# Outlier and Missing Value Visualizations
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.boxplot(ax=axes[0], x=df['AMT_INCOME_TOTAL'], color='lightsalmon')
axes[0].set_title('Annual Income Boxplot (Outlier Detection)')

# Missing values heatmap
sns.heatmap(ax=axes[1], data=df.isnull(), cbar=False, yticklabels=False, cmap='viridis')
axes[1].set_title('Missing Values Heatmap (Yellow shows nulls)')

plt.tight_layout()
plt.show()

In [ ]:
# Step 4 & 5: Data Cleaning & Feature Engineering
print("Cleaning datasets and engineering features...")

# 1. Resolve occupation null values
df['OCCUPATION_TYPE'] = df['OCCUPATION_TYPE'].fillna('Unknown')

# 2. Remove duplicate rows
df.drop_duplicates(inplace=True)

# 3. Drop outliers in Income (using IQR method)
Q1 = df['AMT_INCOME_TOTAL'].quantile(0.25)
Q3 = df['AMT_INCOME_TOTAL'].quantile(0.75)
IQR = Q3 - Q1
upper_limit = Q3 + 3.0 * IQR # 3 * IQR to exclude extreme outliers
df_clean = df[df['AMT_INCOME_TOTAL'] <= upper_limit].copy()

# 4. Feature engineering
# - Income per member
df_clean['INCOME_PER_MEMBER'] = df_clean['AMT_INCOME_TOTAL'] / df_clean['CNT_FAM_MEMBERS']

# - Income Category
def get_income_cat(inc):
    if inc < 100000: return 'Low'
    elif inc < 200000: return 'Medium'
    else: return 'High'
df_clean['INCOME_CAT'] = df_clean['AMT_INCOME_TOTAL'].apply(get_income_cat)

# - Age Category
def get_age_cat(age):
    if age < 30: return 'Young'
    elif age < 50: return 'Middle-Aged'
    else: return 'Senior'
df_clean['AGE_CAT'] = df_clean['AGE'].apply(get_age_cat)

# - Employment Category
def get_employed_cat(years):
    if years == 0: return 'Unemployed'
    elif years < 5: return 'Junior'
    elif years < 15: return 'Mid-level'
    else: return 'Senior'
df_clean['EMPLOYED_CAT'] = df_clean['YEARS_EMPLOYED'].apply(get_employed_cat)

# Drop redundant and mobile variables
df_clean.drop(columns=['DAYS_BIRTH', 'DAYS_EMPLOYED', 'FLAG_MOBIL'], inplace=True, errors='ignore')
print(f"Cleaning complete. Final dataset shape: {df_clean.shape}")

In [ ]:
# Label Encoding & Feature Scaling
from sklearn.preprocessing import LabelEncoder, StandardScaler

cat_cols = ['CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_INCOME_TYPE', 
            'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 
            'OCCUPATION_TYPE', 'INCOME_CAT', 'AGE_CAT', 'EMPLOYED_CAT']

encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))
    encoders[col] = le

scaler = StandardScaler()
num_cols = ['CNT_CHILDREN', 'AMT_INCOME_TOTAL', 'CNT_FAM_MEMBERS', 'AGE', 'YEARS_EMPLOYED', 'INCOME_PER_MEMBER']
df_clean[num_cols] = scaler.fit_transform(df_clean[num_cols])

print("Encodings and scaling finished successfully.")

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(12, 10))
corr = df_clean.drop(columns=['ID'], errors='ignore').corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", cbar=True)
plt.title('Feature Correlation Heatmap')
plt.show()

In [ ]:
# Step 6: Train/Test Split
from sklearn.model_selection import train_test_split

X = df_clean.drop(columns=['ID', 'target'], errors='ignore')
y = df_clean['target']

# 80/20 train/test split, stratify class distributions due to imbalance
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}, y_test shape: {y_test.shape}")

In [ ]:
# Step 7: Train Multiple Machine Learning Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from imblearn.ensemble import BalancedRandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
    'Decision Tree': DecisionTreeClassifier(random_state=42, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(random_state=42, class_weight='balanced'),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss'),
    'Balanced Random Forest': BalancedRandomForestClassifier(random_state=42, sampling_strategy='auto')
}

results = {}

for name, clf in models.items():
    print(f"Training {name}...")
    clf.fit(X_train, y_train)
    
    y_pred = clf.predict(X_test)
    y_prob = clf.predict_proba(X_test)[:, 1] if hasattr(clf, "predict_proba") else np.zeros(len(y_test))
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    auc = roc_auc_score(y_test, y_prob)
    
    results[name] = {
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1 Score': f1,
        'ROC AUC': auc
    }
    print(f"  Accuracy: {acc*100:.2f}%, F1 Score: {f1*100:.2f}%")

results_df = pd.DataFrame(results).T
print("\n--- Model Performance Comparison ---")
print(results_df)

In [ ]:
# Step 8: Choose and Save the Best Model
import joblib

# Pick the best model based on accuracy
best_model_name = results_df['Accuracy'].idxmax()
best_model = models[best_model_name]

print(f"Best model chosen: {best_model_name} with Accuracy {results_df.loc[best_model_name, 'Accuracy']*100:.2f}%")
print("\nBest Model Performance Report:")
print(classification_report(y_test, best_model.predict(X_test)))

# Save best model and Preprocessing tools
os.makedirs('models', exist_ok=True)
joblib.dump(best_model, 'models/card_model.joblib')
joblib.dump(scaler, 'models/scaler.joblib')
joblib.dump(encoders, 'models/label_encoders.joblib')
joblib.dump(list(X.columns), 'models/feature_names.joblib')

# Also save feature importances if the best model has them
if hasattr(best_model, "feature_importances_"):
    importances = best_model.feature_importances_
    feat_imp = pd.Series(importances, index=X.columns).sort_values(ascending=False)
    print("\nFeature Importances:")
    print(feat_imp)
    joblib.dump(feat_imp, 'models/feature_importances.joblib')

print("\nAll modeling files saved successfully in models/ directory.")